# Integrated Comprehension, Fixation, and Individual Differences

This notebook crosses participant-text comprehension/fixation outcomes with reader measures. It is exploratory and tests whether individual differences moderate the original comprehension-fixation relationship.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests
from IPython.display import display, Markdown

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.integrated.io import load_integrated, predictor_summary
from analysis.integrated.models import fit_fixation_moderation, fit_comprehension_moderation, interaction_summary
from analysis.integrated.plots import plot_skill_quartiles, plot_interaction_means

OUTPUT = ROOT / 'analysis' / 'integrated' / 'output' / 'notebook'
PLOTS = OUTPUT / 'plots'
OUTPUT.mkdir(parents=True, exist_ok=True)
PLOTS.mkdir(parents=True, exist_ok=True)

## Data Join and Missingness

In [ ]:
data = load_integrated('both')
print(f'Participant-text observations: {len(data):,}')
print(f'Participants: {data.participant_id.nunique():,}')
display(predictor_summary(data, ['language_skill_composite', 'z_lextale', 'z_vocab.t2.5', 'z_TOWRE_word', 'z_TOWRE_nonword', 'cognitive_skill', 'motivation_standardized', 'spelling_standardized']))
data.to_csv(OUTPUT / 'integrated_analysis_dataset.csv', index=False)

## Reader-Skill Patterns

In [ ]:
for fig in [plot_skill_quartiles(data, PLOTS), plot_interaction_means(data, PLOTS)]:
    display(fig)
    plt.close(fig)

## Wave-Specific Moderation Models

The interaction between comprehension and language skill is the main test. A negative interaction would be consistent with more skilled readers showing fewer fixations as comprehension improves.

In [ ]:
rows = []
models = {}
for wave, group in data.groupby('wave', observed=True):
    fixation_result, fixation_data, fixation_formula = fit_fixation_moderation(group, 'language_skill_composite')
    reverse_result, reverse_data, reverse_formula = fit_comprehension_moderation(group, 'language_skill_composite')
    models[wave] = {'fixation': fixation_result, 'reverse': reverse_result}
    fixation_effect = interaction_summary(fixation_result, 'language_skill_composite')
    reverse_term = 'fixations_per_word:language_skill_composite'
    reverse_low, reverse_high = reverse_result.conf_int().loc[reverse_term]
    rows.append({'wave': wave, 'fixation_rows': len(fixation_data), 'fixation_interaction': fixation_effect['estimate'], 'fixation_interaction_p': fixation_effect['p_value'], 'fixation_effect_ratio': fixation_effect['exponentiated_effect'], 'reverse_interaction': reverse_result.params[reverse_term], 'reverse_interaction_p': reverse_result.pvalues[reverse_term], 'reverse_odds_ratio': np.exp(reverse_result.params[reverse_term]), 'reverse_ci_low': np.exp(reverse_low), 'reverse_ci_high': np.exp(reverse_high)})
    display(Markdown(f'### Wave {wave}'))
    display(pd.DataFrame(rows[-1:], index=None))
results = pd.DataFrame(rows)
results.to_csv(OUTPUT / 'wave_moderation_results.csv', index=False)
display(results)

## Individual Predictors

The language-skill composite is primary. Other predictors are exploratory and should be interpreted with false-discovery-rate correction and predictor-specific sample sizes.

In [ ]:
predictors = ['z_lextale', 'z_vocab.t2.5', 'z_TOWRE_word', 'z_TOWRE_nonword', 'cognitive_skill', 'motivation_standardized', 'spelling_standardized']
secondary = []
for wave, group in data.groupby('wave', observed=True):
    for predictor in predictors:
        if group[predictor].notna().sum() < 50:
            continue
        result, model_data, _ = fit_fixation_moderation(group, predictor)
        effect = interaction_summary(result, predictor)
        secondary.append({'wave': wave, 'predictor': predictor, 'n_rows': len(model_data), 'interaction': effect['estimate'], 'effect_ratio': effect['exponentiated_effect'], 'p_value': effect['p_value'], 'ci_low': effect['ci_low'], 'ci_high': effect['ci_high']})
secondary_results = pd.DataFrame(secondary)
if not secondary_results.empty:
    secondary_results['p_value_fdr'] = secondary_results.groupby('wave')['p_value'].transform(lambda values: multipletests(values, method='fdr_bh')[1])
secondary_results.to_csv(OUTPUT / 'secondary_predictor_interactions.csv', index=False)
display(secondary_results.sort_values('p_value'))

## Interpretation

Treat the interaction as evidence about reader heterogeneity, not as a replacement for the original hypothesis. A stable interaction across waves and sensitivity analyses would support a nuanced reader-skill explanation; a null or inconsistent interaction would reinforce the conclusion that the original relationship is weak or absent.